In [1]:
library(Seurat)
library(Signac)
library(GenomeInfoDb)
library(EnsDb.Hsapiens.v86)
library(ggplot2)
library(patchwork)
library(hdf5r)
library(future)
library(RColorBrewer)
library(dplyr)
library(Matrix)
library(BSgenome.Hsapiens.UCSC.hg38)
library(glue)
library(harmony)
library(matrixStats)
library(scales)
library(biomaRt)
library(curl)
library(goseq)
library(httr)
library(Scillus)
library(TFBSTools)
library(JASPAR2020)
library(ggridges)
library(ggrepel)
library(ggsignif)
library(qusage)
library(tidyverse)
library(effsize)
library(DESeq2)
httr::set_config(config(ssl_verifypeer = 0L))
set.seed(1234)
setwd("/home/jupyter/scATAC_analysis/edit/snatac-rcc-manuscript")
source("scripts/functions.r")


Attaching SeuratObject

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, intersect, is.unsorted, lapply, Map, mapply,
    match, mget, order, paste, pmax, pmax.int, pmin, pmin.int,
    Position, rank, rbind, Reduce, rownames, sapply, setdiff, sort,
    table, tapply, union, unique, unsplit, which.max, which.min


Loading required package: S4Vectors

Loading required package: stats4


Attaching package: ‘S4Vectors’


The following object is masked from ‘package:utils’:

    findMatches


The following objects are masked from ‘package:base’:

    expand.grid, I, unname


Loading required package: IRanges

Loading required package: ensembldb

Loading required packag

# Table S3

## Sheet A: DAPs between BAP1 mut tumor cells and BAP1 wt tumor cells, adv stage only

Regions significantly associated with BAP1 mutations status, no log FC filtering. 

In [ ]:
markers <- readRDS("processed_data/advstage_tumor_bap1_DAPs.rds")
filtered_markers <- markers %>% filter((p_val_adj < 0.05))
# total pos and neg asssoc peaks
dim(filtered_markers)
filtered_markers$group <- ifelse(filtered_markers$avg_log2FC < 0, "BAP1 WT", "BAP1 pLOF")
filtered_markers[, "row.names"] <- NULL


write.table(filtered_markers, file = "tables/s3a_bap1_ccrcc_diffpeaks.txt", sep = "\t", quote = F, col.names = T, row.names = F)


[1] 13571     7

## Sheet B: DAPs between BAP1 mut tumor cells and BAP1 wt tumor cells, all stages

In [4]:
all_stage_markers <- readRDS("processed_data/allstage_tumor_bap1_DAPs.rds")

all_stage_markers <- all_stage_markers %>% filter((p_val_adj < 0.05))
# total pos and neg asssoc peaks
dim(all_stage_markers)
all_stage_markers$group <- ifelse(all_stage_markers$avg_log2FC < 0, "BAP1 WT", "BAP1 pLOF")
all_stage_markers[, "row.names"] <- NULL


write.table(all_stage_markers, file = "tables/s3b_bap1_ccrcc_diffpeaks_allstages.txt", sep = "\t", quote = F, col.names = T, row.names = F)
head(all_stage_markers)


[1] 16560     6

,p_val,avg_log2FC,pct.1,pct.2,p_val_adj,gene,group
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>
1,0,-3.005541,0.058,0.219,0,chr1-633720-634509,BAP1 WT
2,0,4.323645,0.212,0.010,0,chr1-5296725-5298329,BAP1 pLOF
3,0,1.792190,0.170,0.046,0,chr1-16623829-16624466,BAP1 pLOF
4,0,2.051107,0.196,0.042,0,chr1-55442584-55444168,BAP1 pLOF
5,0,1.767395,0.271,0.073,0,chr1-109820986-109822694,BAP1 pLOF
6,0,2.746937,0.178,0.023,0,chr1-167697175-167698570,BAP1 pLOF


## Sheet C: Enriched pathways in BAP1 mutation associated peak set using GREAT

In [10]:
great_results <- readRDS("processed_data/BAP1_tumor_DAP_GREAT_pathway_results.rds")
great_results


Ontology,ID,Desc,BinomRank,BinomP,BinomBonfP,BinomFdrQ,RegionFoldEnrich,ExpRegions,ObsRegions,⋯,HyperRank,HyperP,HyperBonfP,HyperFdrQ,GeneFoldEnrich,ExpGenes,ObsGenes,TotalGenes,GeneSetCov,TermCov
<chr>,<chr>,<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,⋯,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<int>,<dbl>,<dbl>
GO Biological Process,GO:0060337,type I interferon-mediated signaling pathway,2,1.506053e-08,0.0001572319,7.861597e-05,5.116779,3.7132740,19,⋯,1782,1.212223e-01,1.0000000,0.710191300,1.554544,5.7894800,9,64,0.0055147060,0.1406250
GO Biological Process,GO:0071357,cellular response to type I interferon,3,1.523016e-08,0.0001590029,5.300096e-05,5.113094,3.7159490,19,⋯,1875,1.300918e-01,1.0000000,0.724351100,1.530628,5.8799400,9,65,0.0055147060,0.1384615
GO Biological Process,GO:0034340,response to type I interferon,4,1.836392e-08,0.0001917193,4.792983e-05,5.051852,3.7609970,19,⋯,1911,1.392929e-01,1.0000000,0.760972200,1.507436,5.9704010,9,66,0.0055147060,0.1363636
GO Biological Process,GO:0003104,positive regulation of glomerular filtration,5,1.959790e-08,0.0002046021,4.092042e-05,25.193980,0.2778442,7,⋯,702,2.305766e-02,1.0000000,0.342908800,7.369690,0.2713819,2,3,0.0012254900,0.6666667
GO Biological Process,GO:0002335,mature B cell differentiation,8,3.740177e-08,0.0003904745,4.880931e-05,11.231350,0.8903649,10,⋯,1068,5.468656e-02,1.0000000,0.534576500,3.316360,0.9046062,3,10,0.0018382350,0.3000000
GO Biological Process,GO:0003093,regulation of glomerular filtration,9,6.362473e-08,0.0006642422,7.380469e-05,12.714270,0.7078661,9,⋯,918,4.097267e-02,1.0000000,0.465963700,3.684845,0.8141456,3,9,0.0018382350,0.3333333
GO Biological Process,GO:0035455,response to interferon-alpha,12,1.581293e-07,0.0016508700,1.375725e-04,8.310803,1.3235790,11,⋯,2439,1.946947e-01,1.0000000,0.833379500,1.950800,1.5378300,3,17,0.0018382350,0.1764706
GO Biological Process,GO:0044387,negative regulation of protein kinase activity by regulation of protein phosphorylation,13,1.656943e-07,0.0017298480,1.330653e-04,18.334410,0.3817958,7,⋯,961,4.335949e-02,1.0000000,0.471043800,5.527267,0.3618425,2,4,0.0012254900,0.5000000
GO Biological Process,GO:0060333,interferon-gamma-mediated signaling pathway,15,3.547965e-07,0.0037040750,2.469384e-04,3.666618,6.0000800,22,⋯,314,2.752287e-03,1.0000000,0.091509160,2.355884,5.5180980,13,61,0.0079656860,0.2131148


In [11]:
write.table(great_results, "tables/s3c_BAP1_pLOF_GREAT_GOBP_output.txt", sep = "\t", quote = F, row.names = F, col.names = T)


## Sheet D: TF binding site motif enrichment in BAP1 mutation associated peaks

In [8]:
enriched_motifs = readRDS("processed_data/BAP1_tfmotifs.rds")

write.table(enriched_motifs %>% filter(p.adjust < 0.05),
    file = "tables/s3d_BAP1_tfmotifs.txt", sep = "\t", quote = F, row.names = F, col.names = T
)


## Sheet E: GSEA of RNA-seq data from BAP1 knockout vs wild type 786O cells

In [16]:
gsea = readRDS('processed_data/gsea_bap1ko_wt_7860_rna.rds')
write.table(gsea, file = 'tables/s3e_7686o_rna_gsea.txt', sep = '\t', quote = F, row.names = F, col.names = T)

## Sheet F: ORA of DEPs from BAP1 KO vs WT 786-O cells

In [9]:
ora_all = readRDS('processed_data/ora_bap1ko_wt_7860_protein.rds')
ora_supp <- ora_all %>% dplyr::select(-c(ID, log10padj, geneRatio_raw, geneRatio))
write.table(ora_supp, "tables/s3f_ora_bap1ko_wt_7860_protein.txt", sep = "\t", quote = F, row.names = F, col.names = T)


# Export to Excel

In [1]:
import pandas as pd
import os
os.chdir('/home/jupyter/scATAC_analysis/edit/snatac-rcc-manuscript')

In [5]:
bap1_daps = pd.read_csv("tables/s3a_bap1_ccrcc_diffpeaks.txt", sep = "\t")
bap1_daps.head()

bap1_daps_allstage = pd.read_csv("tables/s3b_bap1_ccrcc_diffpeaks_allstages.txt", sep = "\t")
bap1_daps_allstage.head()

bap1_plof_pathways = pd.read_csv("tables/s3c_BAP1_pLOF_GREAT_GOBP_output.txt", sep = "\t")
bap1_plof_pathways.head()

bap1_plof_tf_motifs = pd.read_csv("tables/s3d_BAP1_tfmotifs.txt", sep = "\t")
bap1_plof_tf_motifs.head()

gsea = pd.read_csv("tables/s3e_7686o_rna_gsea.txt", sep = "\t")
gsea = gsea.loc[gsea['pathway'] != 'type-I inducible tumor']
gsea.loc[gsea['pathway'] == '(Bi et al)', 'pathway'] = 'type-I inducible tumor (Bi et al)'
gsea.head()

ora = pd.read_csv("tables/s3f_ora_bap1ko_wt_7860_protein.txt", sep = "\t")
ora = ora.loc[ora['Description'] != 'type-I inducible tumor']
ora.loc[ora['Description'] == '(Bi et al)', 'Description'] = 'type-I inducible tumor (Bi et al)'
ora.head()


,Description,GeneRatio,BgRatio,pvalue,p.adjust,qvalue,geneID,Count,Direction
0,INTERFERON_ALPHA_RESPONSE,27/312,61/2390,1.127301e-09,5.749236e-08,4.746531e-08,IFI44/HLA-C/HERC6/OASL/SP110/DDX60/NMI/EIF2AK2...,27.0,KO
1,INTERFERON_GAMMA_RESPONSE,34/312,98/2390,1.629177e-08,2.923842e-07,2.413904e-07,IFI44/HERC6/OASL/SP110/TNFAIP2/OAS3/DDX60/NMI/...,34.0,KO
3,type-I inducible tumor (Bi et al),29/312,76/2390,1.719907e-08,2.923842e-07,2.413904e-07,IFI44/HLA-C/HERC6/OASL/SP110/DDX58/OAS3/DDX60/...,29.0,KO
4,FATTY_ACID_METABOLISM,29/312,108/2390,6.426785e-05,8.194151e-04,6.765037e-04,IL4I1/ECH1/MDH1/EHHADH/SUCLA2/CA2/ACO2/HADHB/S...,29.0,KO
5,ESTROGEN_RESPONSE_LATE,22/312,84/2390,7.382326e-04,7.529973e-03,6.216696e-03,FARP1/IDH2/ETFB/CCND1/IMPA2/WFS1/DNAJC1/CA2/TP...,22.0,KO


In [6]:
with pd.ExcelWriter('tables/table_S3_draft.xlsx') as writer:  
    bap1_daps.to_excel(writer, sheet_name='A', index = False)
    bap1_daps_allstage.to_excel(writer, sheet_name='B', index = False)
    bap1_plof_pathways.to_excel(writer, sheet_name='C', index = False)
    bap1_plof_tf_motifs.to_excel(writer, sheet_name='D', index = False)
    gsea.to_excel(writer, sheet_name='E', index = False)
    ora.to_excel(writer, sheet_name='F', index = False)